# 02. Contact·Routing·Failover 실습

목표: 시간에 따라 생기고 사라지는 링크를 contact로 모델링하고 earliest-arrival route, hysteresis failover, onboard buffer를 계산한다.

In [1]:
from dataclasses import dataclass
import heapq

@dataclass(frozen=True)
class Contact:
    src: str
    dst: str
    start_s: float
    end_s: float
    rate_bps: float
    one_way_s: float
    confidence: float = 1.0

    def usable_bytes(self, acquisition_s=0, efficiency=1.0):
        duration = max(0.0, self.end_s - self.start_s - acquisition_s)
        return duration * self.rate_bps * efficiency / 8

contacts = [
    Contact('SAT-A', 'SAT-B', 0, 80, 2e6, 0.006, 0.99),
    Contact('SAT-B', 'GROUND', 30, 120, 5e6, 0.004, 0.95),
    Contact('SAT-A', 'SAT-C', 10, 100, 1e6, 0.008, 0.999),
    Contact('SAT-C', 'GROUND', 100, 200, 10e6, 0.005, 0.99),
]
for c in contacts:
    print(c.src, '->', c.dst, f'{c.usable_bytes(acquisition_s=5, efficiency=.8)/1e6:.2f} MB')

SAT-A -> SAT-B 15.00 MB
SAT-B -> GROUND 42.50 MB
SAT-A -> SAT-C 8.50 MB
SAT-C -> GROUND 95.00 MB


In [2]:
def earliest_arrival_route(contacts, source, destination, start_time=0.0):
    # 각 contact에서 현재 도착 시각보다 늦은 시작을 기다릴 수 있다고 가정한다.
    graph = {}
    for contact in contacts:
        graph.setdefault(contact.src, []).append(contact)
    best = {source: start_time}
    previous = {}
    queue = [(start_time, source)]
    while queue:
        current_time, node = heapq.heappop(queue)
        if current_time != best[node]:
            continue
        if node == destination:
            break
        for contact in graph.get(node, []):
            tx_start = max(current_time, contact.start_s)
            if tx_start >= contact.end_s:
                continue
            arrival = tx_start + contact.one_way_s
            if arrival < best.get(contact.dst, float('inf')):
                best[contact.dst] = arrival
                previous[contact.dst] = (node, contact)
                heapq.heappush(queue, (arrival, contact.dst))
    if destination not in best:
        return None
    path, cursor = [], destination
    while cursor != source:
        parent, contact = previous[cursor]
        path.append(contact)
        cursor = parent
    return list(reversed(path)), best[destination]

path, arrival = earliest_arrival_route(contacts, 'SAT-A', 'GROUND')
print(' -> '.join([path[0].src] + [c.dst for c in path]), 'arrival:', arrival)
assert [c.dst for c in path] == ['SAT-B', 'GROUND']

SAT-A -> SAT-B -> GROUND arrival: 30.004


In [3]:
class FailoverController:
    def __init__(self, bad_threshold=3, good_threshold=4, margin_db=2.0):
        self.path = 'PRIMARY'
        self.bad_count = 0
        self.good_count = 0
        self.bad_threshold = bad_threshold
        self.good_threshold = good_threshold
        self.margin_db = margin_db

    def update(self, primary_margin, backup_margin):
        # 여러 sample을 요구해 한 번의 noise spike에 전환하지 않는다.
        if self.path == 'PRIMARY':
            self.bad_count = self.bad_count + 1 if primary_margin < 0 else 0
            if self.bad_count >= self.bad_threshold and backup_margin > self.margin_db:
                self.path, self.bad_count = 'BACKUP', 0
        else:
            self.good_count = self.good_count + 1 if primary_margin > backup_margin + self.margin_db else 0
            if self.good_count >= self.good_threshold:
                self.path, self.good_count = 'PRIMARY', 0
        return self.path

primary = [4, 1, -1, -2, -3, 1, 5, 6, 7, 8, 9]
backup =  [3, 3,  3,  3,  3, 3, 3, 3, 3, 3, 3]
controller = FailoverController()
for index, values in enumerate(zip(primary, backup)):
    print(index, values, controller.update(*values))

0 (4, 3) PRIMARY
1 (1, 3) PRIMARY
2 (-1, 3) PRIMARY
3 (-2, 3) PRIMARY
4 (-3, 3) BACKUP
5 (1, 3) BACKUP
6 (5, 3) BACKUP
7 (6, 3) BACKUP
8 (7, 3) BACKUP
9 (8, 3) BACKUP
10 (9, 3) PRIMARY


In [4]:
def simulate_buffer(seconds, production_bps, contacts, capacity_bytes, efficiency=0.8):
    backlog = 0.0
    dropped = 0.0
    high_water = 0.0
    for t in range(seconds):
        backlog += production_bps / 8
        active_rate = sum(c.rate_bps for c in contacts if c.start_s <= t < c.end_s)
        backlog = max(0.0, backlog - active_rate * efficiency / 8)
        if backlog > capacity_bytes:
            dropped += backlog - capacity_bytes
            backlog = capacity_bytes
        high_water = max(high_water, backlog)
    return {'final_bytes': backlog, 'high_water_bytes': high_water, 'dropped_bytes': dropped}

downlinks = [Contact('SAT-A', 'GROUND', 40, 70, 2e6, 0.003),
             Contact('SAT-A', 'GROUND', 140, 170, 2e6, 0.003)]
result = simulate_buffer(240, 500_000, downlinks, capacity_bytes=12_000_000)
print({k: round(v/1e6, 3) for k, v in result.items()}, 'MB')

{'final_bytes': 4.625, 'high_water_bytes': 4.625, 'dropped_bytes': 0.0} MB


## 확장 과제

- contact capacity가 bundle 크기보다 작으면 route에서 제외하도록 바꾼다.
- confidence와 energy cost를 route cost에 넣고 earliest-arrival 결과와 비교한다.
- failover controller에 최소 체류 시간, acquisition time, stale sample timestamp를 추가한다.
- buffer를 critical/realtime/bulk 세 queue로 나누고 reserved capacity와 aging을 구현한다.